# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Miguel Ángel García Roig  <br>
Url: https://github.com/magarciaroig/MIAR_algortimos_optimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos_Opt_(Miguel_Angel_Garcia_Roig).ipynb<br>
Google Colab: https://colab.research.google.com/drive/1X5o1do7xYcL9mjY08hCavffuSltWfhFD?usp=sharing <br>
Problema:
* **Organizar los horarios de partidos de una jornada de La Liga**

**Descripción del problema:**

Desde la La Liga de fútbol profesional se pretende organizar los horarios de los
partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a
diseñar un algoritmo que realice **la asignación de los partidos a los horarios de forma
que maximice la audiencia**.

Los horarios disponibles se conocen a priori y son los siguientes:
* Viernes: 20h
* Sábado: 12h, 16h, 18h, 20h
* Domingo: 12h, 16h, 18h, 20h
* Lunes: 20h

Cálculo de a audiencia estimada:

* En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores( que tiene relación directa con la audiencia). Hay 3 equipos en la categoría A, 11 equipos de categoría B y 6 equipos de categoría C.
* Se conoce estadísticamente la audiencia que genera cada partido según los equipos
que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos).

| | Viernes | Sábado | Domingo | Lunes |
| :--- | :---: | :---: | :---: | :---: |
| **12h** | - | 0.55 | 0.45 | - |
| **16h** | - | 0.7 | 0.75 | - |
| **18h** | - | 0.8 | 0.85 | - |
| **20h** | 0.4 | 1 | 1 | 0.4 |

* Es posible la coincidencia de horarios pero en este
caso la audiencia de cada partido se verá afectada y
se estima que se reduce en porcentaje según la
siguiente tabla dependiendo del número de
coincidencias:

| Coincidencias | -% |
| :---: | :---: |
| 0 | 0% |
| 1 | 25% |
| 2 | 45% |
| 3 | 60% |
| 4 | 70% |
| 5 | 75% |
| 6 | 78% |
| 7 | 80% |
| 8 | 80% |

Los datos de la jornada de liga en concreto a optimizar se encuentran en la siguiente tabla:

| Partido | Categorías | Horario | Base (Mill.) | Ponderación | Base*Ponderación | Corrección Coincidencia |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| Celta - Real Madrid | B-A | V20 | 1,3 | 0,4 | 0,52 | 0,52 |
| Valencia - R. Sociedad | B-A | S12 | 1,3 | 0,55 | 0,72 | 0,72 |
| Mallorca - Eibar | C-C | S16 | 0,47 | 0,7 | 0,33 | 0,33 |
| Athletic - Barcelona | B-A | S18 | 1,3 | 0,8 | 1,04 | 1,04 |
| Leganés - Osasuna | C-C | S20 | 0,47 | 1 | 0,47 | 0,47 |
| Villarreal - Granada | B-C | D16 | 0,75 | 0,75 | 0,56 | 0,42 |
| Alavés - Levante | B-B | D16 | 0,9 | 0,75 | 0,68 | 0,51 |
| Espanyol - Sevilla | B-B | D18 | 0,9 | 0,85 | 0,77 | 0,77 |
| Betis - Valladolid | B-C | D20 | 0,75 | 1 | 0,75 | 0,75 |
| Atlético - Getafe | B-B | L20 | 0,9 | 0,4 | 0,36 | 0,36 |
....







                                        

# Imports generales e inicialización

In [ ]:
!pip install tqdm

In [ ]:
from tqdm.notebook import tqdm
import random
import itertools
import logging

LOGGER_FILE = 'prac_alg_opt_garciaroig.log'

logger = logging.getLogger(__name__)

logger.propagate = False # evitar que los mensajes de log se propaguen el logger raiz del notebook y los mensajes se duliquen

if logger.hasHandlers():
    logger.handlers.clear()

logger.info('setup completed')

#Modelo

## ¿Como represento el espacio de soluciones?

He eligido una solución basada en algoritmos genéticos. En este contexto el
modelo se configura de la siguiente manera:

* **Genotipo (El Cromosoma)**: Se representa como una lista unidimensional de tamaño 10, donde cada posición corresponde a uno de los 10 partidos de la jornada.
* **Los Genes**: El valor contenido en cada posición es un número entero (del 0 al 9) que sirve como índice para identificar uno de los 10 horarios disponibles (Viernes 20h, Sábado 12h, Sábado 16h, etc.).
* De esta manera, una posible solución completa (un individuo de la población) sería un array como [0, 4, 2, ... ], que indica que el Partido 1 va el Viernes a las 20h (índice 0), el Partido 2 va el Sábado a las 20h (índice 4), etc.

Entrando en más detalles, se configuran las siguientes estructuras de datos para
definir tanto la configuración del problema, como de las restricciones a
implementar:

- **partidos**: Lista de partidos de esta jornada a optimizar. Nótese que, en vez
de codificar partidos concretos (por ejemplo, Celta vs Real Madrid), codificamos
la categoría de los equipos que se enfrentan en cada partido, que es lo realmente relevante para la resolución del problema.
- **audiencia_base**: Diccionario con la audiencia base por tipo de equipos que
se enfrentan en cada partido.
- **slots**: Identificadores de horarios de asignación de partidos
- **coef_horario**: Coeficientes de audiencia base por horario
- **penalizacion_coincidencia**: Penalización en la audiencia por número de partidos coincidentes en el mismo horario
- **hiperparams**: Rango de cada parámetro a considerar para el algoritmo genético, como las poblaciones a generar, el número de generaciones, etc. Hay un paso de optimización que ejecuta el algortimo con todas las configuraciones para determinar los mejores para resolver esta istancia de problema en concreto.


## ¿Cual es la función objetivo?

La función objetivo persigue maximizar la audiencia total de la jornada.

Se calcula iterando sobre los 10 partidos y sumando la audiencia individual de cada uno, la cual es el
producto de tres variables:

$$Fitness = \sum_{i=1}^{10} (AudienciaBase_{i} \times CoefHorario_{i} \times CoefCoincidencia_{i})$$

Donde:

* $AudienciaBase$: Depende de la categoría de los equipos enfrentados (ej. A-A = 2 millones, C-C = 0.47 millones).
* $CoefHorario$: Es la ponderación asociada al día y la hora asignada (ej. Sábado 20h = 1; Viernes 20h = 0.4).
* $CoefCoincidencia$: Es el factor de corrección si hay partidos simultáneos. Si un partido coincide en el mismo horario con otro (1 coincidencia), su audiencia se reduce un 25% (se multiplica por 0.75).

Se implementa en la función python `calcular_fitness(cromosoma)`, donde cromosoma es un individuo de la población (lista unidimensional de tamaño 10, donde cada posición corresponde a uno de los 10 partidos de la jornada)

##¿Como implemento las restricciones?

El problema plantea una restricción de obligatoriedad: siempre debe haber al menos un partido el viernes y un
partido el lunes.

En nuestra implementación, esta restricción se maneja combinando dos estrategias para asegurar que
la población  cumpla con las reglas:

1. **Generación Controlada (Siembra Inicial)**: Al construir la población inicial de soluciones candidatas, no dejamos que el 100% del cromosoma se genere al azar. En la función `crear_individuo()`, forzamos explícitamente que una posición del vector (por ejemplo, el primer partido) se asigne al slot del viernes (V20) y otra (el último partido) al slot del lunes (L20). Esto asegura que el algoritmo arranque su búsqueda partiendo de soluciones que ya son factibles.
2. **Penalización Severa en la Función de Evaluación (Fitness)**: Debido a que los operadores genéticos de cruce y mutación  realizan cambios aleatorios, es muy probable que en algún momento modifiquen el horario del viernes o del lunes, rompiendo la restricción. Para solucionarlo, en la función `calcular_fitness()`, contamos los partidos asignados a cada día. Si detectamos que un cromosoma ya no tiene partidos el viernes o el lunes, le aplicamos una penalización drástica (devolviendo un valor de audiencia casi nulo de 0.1).

Al usar esta función de evaluación para medir la calidad de los individuos, el mecanismo de selección natural del algoritmo identificará estas soluciones "ilegales" como los individuos peor adaptados, eliminándolos de la población y asegurando que no transmitan su configuración defectuosa a la siguiente generación.

In [ ]:
# Categorías de los 10 partidos de la jornada
partidos = [
    ('B', 'A'), ('B', 'A'), ('C', 'C'), ('B', 'A'), ('C', 'C'),
    ('B', 'C'), ('B', 'B'), ('B', 'B'), ('B', 'C'), ('B', 'B')
]

# Audiencia base según combinación de categorías
audiencia_base = {
    ('A', 'A'): 2.0, ('A', 'B'): 1.3, ('B', 'A'): 1.3,
    ('A', 'C'): 1.0, ('C', 'A'): 1.0, ('B', 'B'): 0.9,
    ('B', 'C'): 0.75, ('C', 'B'): 0.75, ('C', 'C'): 0.47
}

# Identificadores de horarios de asignación de partidos
# Uno para cada combinación día-hora (p.e. V20 significa 'Viernes 20 horas')
slots = [
    'V20', 'S12', 'S16', 'S18', 'S20', 'D12', 'D16', 'D18', 'D20', 'L20'
]

# Coeficientes de audiencias base por horario
coef_horario = {
    'V20': 0.4, 'S12': 0.55, 'S16': 0.7, 'S18': 0.8, 'S20': 1.0,
    'D12': 0.45, 'D16': 0.75, 'D18': 0.85, 'D20': 1.0, 'L20': 0.4
}

# Penalizaciones por coincidencias de varios partidos en el mismo horario
# Formato: {número_de_coincidencias: factor_multiplicador}
penalizacion_coincidencia = {
    0: 1.0, 1: 0.75, 2: 0.55, 3: 0.4, 4: 0.3, 5: 0.25, 6: 0.22, 7: 0.2, 8: 0.2
}

# Rangos de hiperparámetros que vamos a probar para nuestro algoritmo (genético
# en este caso). Probaremos varias combinaciones cruzadas
# para tratar de encontrar los mejores para nuestro problema, y acercarnos así
# más al óptimo

HIPERPARAMS = {
  'max_poblacion': [100, 200, 300],
  'generaciones': [200, 500],
  'prob_mutacion': [0.1, 0.2, 0.3],
  'pct_elite': [0.05, 0.10],
  'pct_sel_padres': [0.2, 0.3, 0.5]
}


#Diseño

## ¿Que técnica utilizo?

Para resolver el problema de asignación de horarios de La Liga, se utiliza un Algoritmo Genético, el cual pertenece a la
familia de los métodos heurísticos y metaheurísticos basados en la evolución poblacional.

# ¿Por qué?

La elección de esta técnica se justifica por los siguientes puntos clave:

* **Tamaño del espacio de soluciones y no linealidad**: El problema cuenta con un espacio de búsqueda inmenso (10.000 millones de combinaciones posibles). En la teoría, los algoritmos genéticos y evolutivos se utilizan precisamente cuando el espacio de soluciones es grande y tanto este como la función objetivo no presentan un comportamiento lineal. En nuestro caso, la función objetivo (audiencia) es fuertemente no lineal debido a las penalizaciones porcentuales variables que ocurren cuando los horarios coinciden.
* **Inviabilidad de los métodos exactos**: Al ser un problema de optimización combinatoria de gran magnitud, intentar buscar la solución mediante fuerza bruta o métodos exactos requeriría un tiempo de ejecución exponencial, haciéndolo inabordable en la práctica. Los métodos metaheurísticos, en cambio, están diseñados para encontrar "buenas soluciones" (muy cercanas al óptimo global) en tiempos de cómputo muy cortos.
* **Facilidad de representación (Genotipo/Fenotipo)**: Para aplicar un algoritmo genético, debemos encontrar una representación a través de las variables del algoritmo (el genotipo) de las soluciones del problema (el fenotipo) . Este problema se modela de forma muy natural usando un vector numérico donde cada posición representa un partido y el valor contenido representa el horario asignado.
* **Capacidad para escapar de óptimos locales**: Una dificultad habitual en la optimización es quedarse atrapado en un "óptimo local" (una configuración que parece buena, pero no es la mejor absoluta) . Los algoritmos genéticos evitan este estancamiento mediante la mutación (alteración individual aleatoria de un horario) y el cruce (combinación de configuraciones entre distintas jornadas) . Esto diversifica la búsqueda.
* **Mejora iterativa basada en la adaptación (Fitness)**: El algoritmo utiliza una función de evaluación (fitness) que en este caso es el cálculo total de la audiencia . A través del proceso de selección, el algoritmo favorece probabilísticamente a los calendarios con mayor audiencia para que transmitan su "contenido genético" a las siguientes generaciones, asegurando una convergencia iterativa hacia la mejor solución posible.


In [ ]:
def calcular_fitness(cromosoma):
  """
  La función fitness calculará la audiencia potencial.
  Se aplicán las restricciones definidas en el problema.
  """
  total_audiencia = 0
  conteo_slots = {slot: 0 for slot in slots}

  # Contar cuántos partidos hay en cada slot para aplicar penalizaciones
  for gen in cromosoma:
      conteo_slots[slots[gen]] += 1

  # Calcular audiencia partido a partido
  for i, gen in enumerate(cromosoma):
    slot_actual = slots[gen]
    base = audiencia_base[partidos[i]]
    ponderacion = coef_horario[slot_actual]

    # Las coincidencias son n-1 (si hay 2 partidos, hay 1 coincidencia)
    reduccion = 1
    num_coincidencias = conteo_slots[slot_actual] - 1
    if num_coincidencias > -1:
        reduccion = penalizacion_coincidencia.get(num_coincidencias, 0.2)

    total_audiencia += base * ponderacion * reduccion

  # Penalización severa si no hay partidos en viernes o lunes
  if conteo_slots['V20'] == 0 or conteo_slots['L20'] == 0:
      return 0.1  # Valor mínimo para desincentivar la solución

  return total_audiencia

def indice_horario(cod_horario):
  """
  Función auxiliar de conveniencia para obtener el índice de un horario en concreto.
  Por ejemplo 'V20'-> 0
  """
  for idx, cod in enumerate(slots):
      if cod_horario.upper() in cod.upper():
          return idx

  return -1

def crear_individuo():
  """
  Crea un individuo de forma aleatoria: Una lista de índices de slots por partido,
  dónde un valor x en la posición i-ésima de la lista significa que al partido
  i le asignamos el slot x. Un slot es un horario de partido específico
  (por ejemplo 'S16' o 'Sábado a las 16 horas')
  """
  # A cada partido le asignamos un horario elegido al azar
  cromosoma = [random.randint(0, len(slots) - 1) for _ in range(len(partidos))]

  # Para asegurar soluciones que cumplen con las restricciones del problema
  # nos aseguramos los primeros individuos en la generación son válidos
  # (por definición del problema siempre debemos programar al menos
  # un partido el viernes y otro el lunes)
  cromosoma[0] = indice_horario('V')  # Primer partido en viernes
  cromosoma[-1] = indice_horario('L')  # Último partido en lunes

  return cromosoma

def cruce(p1, p2):
  """
  Operador genético de cruce: A partir de dos padres (p1,p2) genera un nuevo individuo
  combinando aleatoriamente genes de ambos padres. Esto es, para algunos partidos
  tendrá la planificación del padre 1, y para otros la del padre 2
  """
  # Verificar que ambos padres tienen la misma dimensión
  if len(p1) != len(p2):
      raise ValueError(
          f"Incompatibilidad genética: El padre 1 tiene {len(p1)} genes, pero el padre 2 tiene {len(p2)}.")

  # Calculamos la longitud dinámicamente basándonos en el tamaño del padre
  tam_cromosoma = len(p1)

  # Elegimos un punto de corte entre el índice 1 y el penúltimo índice.
  merge_idx = random.randint(1, tam_cromosoma - 1)

  # Devolvemos los primeros genes del primer padre, y el resto del segundo
  return p1[:merge_idx] + p2[merge_idx:]


def mutacion(cromosoma, prob_mutacion=0.1):
  """
  Operador genético de mutación: En caso de que el azar así lo dicte, mutamos el cromosoma del individuo,
  cambiamos algún/nos de sus gen/es.

  En nuestro caso elegiremos uno de los partidos de forma aleatoria, y le asignaremos un nuevo
  slot (planificación) de manera aleatoria
  """
  # caso base: no se da una mutacion aleatoria. Devolvemos el mismo individuo
  if random.random() >= prob_mutacion:
    return cromosoma

  # Producimos una mutación aleatoria
  idx_gen_aleatorio = random.randint(0, len(partidos) - 1)
  slot_horario_aleatorio = random.randint(0, len(slots) - 1)
  cromosoma[idx_gen_aleatorio] = slot_horario_aleatorio

  return cromosoma

def planificar_jornada(max_poblacion, generaciones, prob_mutacion, pct_elite, pct_sel_padres):
  """
  Recibe los hiperparámetros de nuestro algoritmo genético y retorna la mejor planificación encontrada
  para la jornada de liga

  :param max_poblacion: El número máximo de individuos (el número de distintas planificaciones de la jornada de liga) a generar
  :param generaciones: Número de iteraciones para conseguir el mejor resultado
  :param prob_mutacion: La probabilidad de que un individuo (una planificación de jornada) mute (uno de sus partidos cambie de horario)
  :param pct_elite: Porcentaje de mejores individuos (elite) que mantenemos en cada generación
  :param pct_sel_padres: Porcentaje de padres de la población que son seleccionados desde los mejores al azar para
  crear nuevos hijos para las siguientes generaciones. Debería ser superior a {pcr_elite}, porque si no solo la elite
  puede generar descendencia para las siguientes generaciones
  """
  poblacion = [crear_individuo() for _ in range(max_poblacion)]

  for gen_idx in range(generaciones):
      # Ordenamos la población en función de los individuos más prometedores
      poblacion = sorted(poblacion, key=lambda x: calcular_fitness(x), reverse=True)

      # Mantenemos el top {pct_elite}% como élite (10 individuos si la población es 200)
      num_elite = int(max_poblacion * pct_elite)
      nueva_poblacion = poblacion[:num_elite]

      # Evolucionamos el resto hasta llegar a {max_poblacion}
      while len(nueva_poblacion) < max_poblacion:
          # Seleccionar padres del top {pct_sel_padres} % de la población actual
          top_padres = int(max_poblacion * pct_sel_padres)
          padre1 = random.choice(poblacion[:top_padres])
          padre2 = random.choice(poblacion[:top_padres])

          # Generamos un nuevo hijo a partir de dos padres
          # y en caso de que el azar lo dicte, le producimos una mutación
          hijo = cruce(padre1, padre2)
          hijo = mutacion(hijo, prob_mutacion)
          nueva_poblacion.append(hijo)

      poblacion = nueva_poblacion

  return poblacion[0]

def mostrar_solucion(solucion):
  """
  Función de utilidad que, dado un individuo o planificación de una jornada, muestra la audiencia potencial
  y la configuración del horario de cada partido de una forma 'human friendly'
  """
  print(f"{calcular_fitness(solucion):.2f} Millones")
  for i, s_idx in enumerate(solucion):
      print(f"Partido {i + 1} ({partidos[i]}): {slots[s_idx]}")

def optimizar_hiperparametros(rango_hiper_params):
  """
  Optimiza los hiperparámetros de nuestro algoritmo genético, generando distintas combinaciones y ejecutando
  el algoritmo con las distintas combinaciones de parámetros.

  Mantiene estadísticas con la mejor solución encontrada, y la muestra al finalizar.
  """

  # Extraer las claves y los valores
  keys = list(rango_hiper_params.keys())
  values = list(rango_hiper_params.values())

  # Generar todas las combinaciones posibles (Producto cartesiano)
  combinaciones = list(itertools.product(*values))
  print(f"Iniciando optimizador de hiperparámetros. Total de combinaciones a probar: {len(combinaciones)}\n")

  mejor_score_promedio = -1
  mejores_parametros = None
  mejor_individuo_global = None

  # Número de veces que se prueba cada combinación para mitigar el factor "suerte"
  num_ejecuciones_por_comb = 3

  for idx, combinacion in tqdm(enumerate(combinaciones)):
        # Mapear los valores a sus respectivos nombres de parámetros
        params = dict(zip(keys, combinacion))

        scores_actuales = []
        mejor_individuo_comb = None
        mejor_score_comb = -1

        # Ejecutamos el algoritmo varias veces para esta combinación
        for _ in range(num_ejecuciones_por_comb):
          # Desempaquetar el diccionario como argumentos de la función (**params)
          individuo = planificar_jornada(**params)
          score = calcular_fitness(individuo)
          scores_actuales.append(score)

          # Guardamos el mejor individuo específico de esta combinación
          if score > mejor_score_comb:
            mejor_score_comb = score
            mejor_individuo_comb = individuo

        # Calcular el rendimiento real promedio de estos parámetros
        score_promedio = sum(scores_actuales) / num_ejecuciones_por_comb

        logger.info(f"[{idx + 1}/{len(combinaciones)}] Params: {params}")
        logger.info(f"   -> Score Medio: {score_promedio:.4f} (Mejor pico: {mejor_score_comb:.4f})")

        # Actualizar el récord global si encontramos un promedio mejor
        if score_promedio > mejor_score_promedio:
          mejor_score_promedio = score_promedio
          mejores_parametros = params
          mejor_individuo_global = mejor_individuo_comb

  print("\n" + "=" * 60)
  print("BÚSQUEDA TERMINADA")
  print("=" * 60)
  print(f"Los Mejores Hiperparámetros encontrados son:")
  for k, v in mejores_parametros.items():
    print(f" - {k}: {v}")

  print(f"\nScore Promedio Esperado: {mejor_score_promedio:.4f} Millones")
  print("\nMejor Planificación Global encontrada con esta configuración:")
  print("-" * 30)
  mostrar_solucion(mejor_individuo_global)


In [ ]:
# Ejecutar (lanzar optimizador hiper-parámetros y printear mejor solución encontrada)
optimizar_hiperparametros(HIPERPARAMS)

Iniciando optimizador de hiperparámetros. Total de combinaciones a probar: 108



0it [00:00, ?it/s]


BÚSQUEDA TERMINADA
Los Mejores Hiperparámetros encontrados son:
 - max_poblacion: 300
 - generaciones: 500
 - prob_mutacion: 0.3
 - pct_elite: 0.1
 - pct_sel_padres: 0.5

Score Promedio Esperado: 6.7695 Millones

Mejor Planificación Global encontrada con esta configuración:
------------------------------
6.80 Millones
Partido 1 (('B', 'A')): D20
Partido 2 (('B', 'A')): S20
Partido 3 (('C', 'C')): V20
Partido 4 (('B', 'A')): D18
Partido 5 (('C', 'C')): D12
Partido 6 (('B', 'C')): S12
Partido 7 (('B', 'B')): S18
Partido 8 (('B', 'B')): D16
Partido 9 (('B', 'C')): S16
Partido 10 (('B', 'B')): L20


#Análisis

## Contabilizar el espacio de soluciones

El espacio de soluciones define exactamente cuántas combinaciones posibles existen para asignar los horarios
a los partidos.

* Según los datos del problema, tenemos 10 partidos diferentes en una jornada.
* Los horarios disponibles son 10 en total (1 el viernes, 4 el sábado, 4 el domingo y 1 el lunes)
* Como es posible la coincidencia de horarios (es decir, asignar varios partidos al mismo hueco horario, asumiendo una penalización), cada uno de los 10 partidos puede ubicarse libremente en cualquiera de los 10 horarios disponibles

Por lo tanto, el tamaño del espacio de soluciones se calcula como las permutaciones con repetición, es decir, el número de opciones de horario elevado al número de partidos:

$Espacio~de~soluciones = 10^{10}$ combinaciones.

Esto significa que existen 10.000 millones de posibles calendarios distintos solo para esta jornada.

## Orden de complejidad

### Para el algoritmo de Fuerza Bruta (Método exacto)

* Orden de complejidad: Sería de orden exponencial, tal que $O(m^n)$; donde $m$ es la cantidad de horarios disponibles y $n$ es el número de partidos.
* Viabilidad: Sería practicamente imposible de llevar a la práctica. Intentar calcular 10.000 millones de opciones es un proceso muy ineficiente y computacionalmente muy costoso.

### Para el Algoritmo Genético (Método Metaheurístico)

* Análisis: Las técnicas genéticas y evolutivas se utilizan precisamente cuando el espacio de soluciones es muy grande y la función objetivo no presenta un comportamiento lineal (como ocurre aquí con las penalizaciones por coincidencia).
* Orden de complejidad: El orden de complejidad ya no está atado a explorar todo el espacio $10^{10}$, sino a los parámetros que se configuran en el diseño del algoritmo. La complejidad computacional se aproxima a $O(G \times P)$, donde $G$ es el número de generaciones (iteraciones) y $P$ es el tamaño de la población (individuos o calendarios evaluados por generación). El paso de optimización de hiperparámetros ha determinado `P=300` y `G=500` con lo que la complejidad sería $O(N)$
* Viabilidad: Los métodos heurísticos y metaheurísticos permiten obtener buenas soluciones en tiempos de cómputo moderadaente acotados, aunque no aseguran matemáticamente que la solución encontrada sea el óptio global. Una complejidad lineal es un excelente resultado, que permitiría aplicar el algoritmo a problemas de cierto tamaño.

En cuanto al detalle del cáclculo del orden de complejidad, lo desarrollamos aquí:


#### Complejidad de las Operaciones Base

* **`crear_individuo()`**: Crea un vector de tamaño $n$ mediante una comprensión de listas y asigna dos valores fijos.
  * **Complejidad:** $O(n)$
* **`calcular_fitness(cromosoma)`**: Recorre el cromosoma de tamaño $n$ dos veces (una para contar los slots y otra para calcular las puntuaciones base y ponderaciones). Los accesos a diccionarios en Python (`audiencia_base`, `coef_horario`) toman tiempo constante $O(1)$.
  * **Complejidad:** $O(n)$
* **`cruce(p1, p2)`**: Genera un número aleatorio $O(1)$ y realiza un *slicing* (corte) y concatenación de dos listas de tamaño $n$. En Python, el *slicing* de listas toma tiempo proporcional a la longitud de la lista.
  * **Complejidad:** $O(n)$
* **`mutacion(cromosoma)`**: Modifica el valor de un índice específico en la lista. El acceso y asignación por índice es constante.
  * **Complejidad:** $O(1)$

#### Complejidad de una Generación completa

Dentro del bucle principal que se repite $G$ veces, ocurren los siguientes pasos:

* **Evaluación y Ordenación** (`sorted(poblacion, key=lambda x: calcular_fitness(x))`):
  * Python usa *Timsort*, cuya complejidad de ordenación es $O(P \log P)$.
  * El parámetro `key` garantiza que `calcular_fitness()` se llame exactamente una vez por cada individuo. Evaluar $P$ individuos toma $P \times O(n) = O(P \cdot n)$.
  * **Coste total del paso:** $O(P \cdot n + P \log P)$
* **Elitismo** (`nueva_poblacion = poblacion[:num_elite]`):
  * Cortar una lista de tamaño $P$ toma a lo sumo $O(P)$.
* **Reproducción** (bucle `while`):
  * Se ejecuta aproximadamente $P$ veces (para rellenar la población).
  * En cada iteración hace: dos selecciones aleatorias $O(1)$, un cruce $O(n)$ y una mutación $O(1)$.
  * **Coste total del paso:** $P \times O(n) = O(P \cdot n)$

**Suma de 1 sola generación:** $O(P \cdot n + P \log P) + O(P) + O(P \cdot n)$.

Eliminando los términos menos significativos (porque $P \cdot n$ crece más rápido que $P$), nos queda que el coste de procesar una generación es:
$$O(P \cdot n + P \log P)$$

#### Complejidad Total del Algoritmo Genético

Como el bucle de generaciones se repite $G$ veces, multiplicamos el coste de una generación por $G$:

$$Complejidad~GA = O(G \cdot P \cdot n + G \cdot P \log P)$$